# 01 · Intermediate tables

Requires a GCP project with billing enabled.

| Step | Table | Reads |
|---|---|---|
| 1 | `spends` | public `inputs` table, one scan |
| 2 | `utxo_life` | public `outputs` table + `spends` |
| 3 | `address_stats`, `supply_by_type` | `utxo_life` only |

`utxo_life` has one row per output: creation height, value, script type, address, spending height and spending transaction. All later tables are derived from it. Transaction hashes are stored as BYTES (32 bytes); use `TO_HEX()` to display them.

In [ ]:
# Configuration
PROJECT_ID = ""            # project with billing enabled
DATASET    = "qbtc_raw"
LOCATION   = "US"          # the public dataset is in US; must match

MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 50.0
PRICE_PER_TIB_USD = 6.25

CONFIRM       = False      # after checking the dry-run cost, set to True and re-run
FORCE_REBUILD = False      # True rebuilds tables that already exist

assert PROJECT_ID, "Set PROJECT_ID first"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, time, datetime as dt
import pandas as pd

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
PUB = "bigquery-public-data.crypto_bitcoin"
DS  = f"{PROJECT_ID}.{DATASET}"

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []
print("Spent so far: $%.2f" % LEDGER["usd_total"])

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 1)}

def build(name, sql):
    """Build one table. Skip it if it exists; save the job_id before waiting."""
    n = table_rows(name)
    if n and not FORCE_REBUILD:
        print(f"[{name}] exists, {n:,} rows; skipped. Set FORCE_REBUILD = True to rebuild")
        return

    gb, cost = dry_run(sql)
    print(f"[{name}] estimated scan {gb:,.1f} GB ~ ${cost:,.2f}")
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] ${cost:.2f} exceeds the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would reach ${LEDGER['usd_total']+cost:.2f}, above the limit ${MAX_USD_TOTAL}")
    if not CONFIRM:
        print(f"[{name}] CONFIRM=False: cost estimate only, not executed")
        return

    cfg = bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30),   # hard cap enforced by BigQuery
        use_query_cache=False,
    )
    job = client.query(sql, job_config=cfg)

    # save before waiting, so the last cell can reconnect after a disconnect
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id}")
    print(f"[{name}] the job runs on the BigQuery server and finishes even if Colab disconnects")

    t0 = time.time()
    job.result()                      # wait for completion
    billed_gb = (job.total_bytes_billed or 0) / 2**30
    spent = usd(billed_gb)
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id,
                           "billed_gb": round(billed_gb, 1), "usd": round(spent, 2),
                           "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + spent, 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, billed {billed_gb:,.1f} GB ~ ${spent:,.2f}, {LEDGER['jobs'][-1]['seconds']}s")
    print(f"[{name}] total ${LEDGER['usd_total']:,.2f} / ${MAX_USD_TOTAL}")
    print(table_info(name))

In [ ]:
# Create the dataset if it does not exist
ds = bigquery.Dataset(DS); ds.location = LOCATION
try:
    client.get_dataset(DS); print("Dataset exists:", DS)
except Exception:
    client.create_dataset(ds); print("Dataset created:", DS)

## Step 1 · `spends`

Spent output, spending height and spending transaction from the public `inputs` table, coinbase inputs excluded. Partitioned by spending month, clustered by spent txid.

In [ ]:
SQL_SPENDS = f"""
CREATE OR REPLACE TABLE `{DS}.spends`
PARTITION BY DATE_TRUNC(spent_date, MONTH)
CLUSTER BY txid
AS
SELECT
  FROM_HEX(spent_transaction_hash) AS txid,
  spent_output_index               AS vout,
  block_number                     AS spent_height,
  DATE(block_timestamp)            AS spent_date,
  FROM_HEX(transaction_hash)       AS spending_txid,
  index                            AS vin
FROM `{PUB}.inputs`
WHERE spent_transaction_hash IS NOT NULL
"""

build("spends", SQL_SPENDS)

## Step 2 · `utxo_life`

Public `outputs` left-joined to `spends`, excluding OP_RETURN outputs (268 million rows). Full address arrays of multisig outputs are kept.

In [ ]:
SQL_UTXO_LIFE = f"""
CREATE OR REPLACE TABLE `{DS}.utxo_life`
PARTITION BY DATE_TRUNC(created_date, MONTH)
CLUSTER BY address
AS
WITH o AS (
  SELECT
    FROM_HEX(transaction_hash)    AS txid,
    index                         AS vout,
    block_number                  AS created_height,
    DATE(block_timestamp)         AS created_date,
    CAST(value AS INT64)          AS sats,
    type,
    addresses[SAFE_OFFSET(0)]     AS address,
    ARRAY_LENGTH(addresses)       AS n_addresses,
    IF(type = 'multisig', addresses, NULL) AS multisig_addresses
  FROM `{PUB}.outputs`
  WHERE type != 'nulldata'
)
SELECT
  o.txid, o.vout, o.created_height, o.created_date, o.sats, o.type,
  o.address, o.n_addresses, o.multisig_addresses,
  s.spent_height, s.spent_date, s.spending_txid
FROM o
LEFT JOIN `{DS}.spends` s USING (txid, vout)
"""

build("utxo_life", SQL_UTXO_LIFE)

### Fallback for step 2

If step 2 fails with "Resources exceeded", this cell builds the table in four parts by creation year (a few extra scans of `spends`). Not run by default.

In [ ]:
RUN_CHUNKED_FALLBACK = False   # set to True only if step 2 failed

CHUNKS = [("2009-01-01", "2013-12-31"), ("2014-01-01", "2017-12-31"),
          ("2018-01-01", "2021-12-31"), ("2022-01-01", "2099-12-31")]

if RUN_CHUNKED_FALLBACK and CONFIRM:
    for k, (a, b) in enumerate(CHUNKS):
        name = f"utxo_life_p{k}"
        sql = SQL_UTXO_LIFE.replace(
            f"CREATE OR REPLACE TABLE `{DS}.utxo_life`",
            f"CREATE OR REPLACE TABLE `{DS}.{name}`"
        ).replace(
            "WHERE type != 'nulldata'",
            f"WHERE type != 'nulldata' AND DATE(block_timestamp) BETWEEN DATE('{a}') AND DATE('{b}')"
        )
        build(name, sql)
    # merge the four parts into one table
    union = " UNION ALL ".join([f"SELECT * FROM `{DS}.utxo_life_p{k}`" for k in range(len(CHUNKS))])
    build("utxo_life", f"""
CREATE OR REPLACE TABLE `{DS}.utxo_life`
PARTITION BY DATE_TRUNC(created_date, MONTH) CLUSTER BY address AS
{union}
""")

## Step 3 · `address_stats` and `supply_by_type`

Reads `utxo_life` only. `first_spend_height` marks when the public key behind a hash-type address was first revealed.

In [ ]:
SQL_ADDRESS_STATS = f"""
CREATE OR REPLACE TABLE `{DS}.address_stats`
CLUSTER BY address
AS
SELECT
  address,
  MIN(created_height)                    AS first_seen_height,
  MIN(created_date)                      AS first_seen_date,
  MAX(created_date)                      AS last_received_date,
  MIN(spent_height)                      AS first_spend_height,
  MIN(spent_date)                        AS first_spend_date,
  MAX(spent_height)                      AS last_spend_height,
  MAX(spent_date)                        AS last_spend_date,
  COUNT(*)                               AS n_received,
  COUNTIF(spent_height IS NOT NULL)      AS n_spent,
  COUNTIF(spent_height IS NULL)          AS n_unspent,
  SUM(sats)                              AS sats_received,
  SUM(IF(spent_height IS NULL, sats, 0)) AS sats_unspent,
  ARRAY_AGG(DISTINCT type ORDER BY type) AS types
FROM `{DS}.utxo_life`
WHERE address IS NOT NULL
GROUP BY address
"""

build("address_stats", SQL_ADDRESS_STATS)

In [ ]:
SQL_SUPPLY = f"""
CREATE OR REPLACE TABLE `{DS}.supply_by_type` AS
SELECT
  type,
  COUNT(*)                AS n_utxo,
  ROUND(SUM(sats)/1e8, 4) AS btc_unspent,
  MIN(created_date)       AS first_date,
  MAX(created_date)       AS last_date
FROM `{DS}.utxo_life`
WHERE spent_height IS NULL
GROUP BY type
"""

build("supply_by_type", SQL_SUPPLY)

## Reconciliation

- Total unspent supply vs theoretical issuance (about 20.09 million BTC in September 2026)
- Unspent `pubkey` outputs (about 1.7 million BTC) vs published estimates (Deloitte, Glassnode)
- Unspent `witness_v1_taproot` outputs vs the figure in Google's white paper

In [ ]:
supply = client.query(f"SELECT * FROM `{DS}.supply_by_type` ORDER BY btc_unspent DESC").to_dataframe()
display(supply)
print(f"Total unspent {supply['btc_unspent'].sum():,.2f} BTC")
print("Reference: theoretical issuance in September 2026 is about 20.09 million BTC (including lost coins)")

In [ ]:
# Reconnect to the most recently submitted jobs
for rec in JOBS[-5:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        print("  waiting ...")
        job.result()
        print("  done")
print()
for t in ["spends", "utxo_life", "address_stats", "supply_by_type"]:
    n = table_rows(t)
    print(f"{t:16s}", f"{n:,} rows" if n else "not built yet")

In [ ]:
report = {
    "generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
    "project": PROJECT_ID, "dataset": DATASET,
    "ledger": LEDGER, "tables": [],
}
for t in ["spends", "utxo_life", "address_stats", "supply_by_type"]:
    try:
        report["tables"].append(table_info(t))
    except Exception as e:
        report["tables"].append({"table": t, "error": str(e)})
try:
    report["supply_by_type"] = supply.astype(str).to_dict(orient="records")
except Exception:
    pass

print(json.dumps(report["tables"], ensure_ascii=False, indent=2))
print("Total spent $%.2f" % LEDGER["usd_total"])

with open("build_report_01.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
try:
    from google.colab import files
    files.download("build_report_01.json")
except Exception:
    print("Report saved to build_report_01.json")